Set up

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_auc_score, roc_curve, balanced_accuracy_score, recall_score,
                             accuracy_score, ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.calibration import calibration_curve
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Find the project folder whether the notebook runs from notebooks/ or from the top folder
HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == "notebooks" else HERE
sys.path.insert(0, str(ROOT / "notebooks"))
os.chdir(ROOT)                       # train_baseline uses paths like data/manifest_split.csv
import train_baseline as tb

SPLIT = "val"                        # change to "test" ONCE, when the model is final
N_BOOT = 2000                        # bootstrap resamples for confidence intervals
RNG = np.random.default_rng(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if SPLIT == "test":
    print("⚠️  Evaluating on the TEST set. Don't change the model after looking at these numbers.")
print("device:", device, "| evaluating on:", SPLIT)

CNN predictions, one row per subject

In [ ]:
manifest = pd.read_csv(tb.MANIFEST_SPLIT)
subject_info = manifest.drop_duplicates("sub_id")[["sub_id", "age", "sex"]]

model = tb.Baseline3DCNN().to(device)
model.load_state_dict(torch.load(tb.CHECKPOINT, map_location=device))

def subject_predictions(split):
    """CNN probability of autism for each subject in a split (scans averaged per subject)."""
    scans = manifest[manifest["split"] == split]
    subj = tb.per_subject(tb.predict(model, scans, batch_size=4, device=device))
    return subj.merge(subject_info, on="sub_id")

val = subject_predictions("val")
ev = val.copy() if SPLIT == "val" else subject_predictions(SPLIT)
print(f"{len(ev)} subjects, {int(ev['label'].sum())} autism, {int((ev['label'] == 0).sum())} control")
ev.head()

Choose the decision threshold on the validation set

In [ ]:
def youden_threshold(y, p):
    """Threshold that maximizes sensitivity + specificity - 1."""
    fpr, tpr, thr = roc_curve(y, p)
    thr = np.minimum(thr, 1.0)               # roc_curve's first threshold is inf
    return float(thr[np.argmax(tpr - fpr)])

THRESHOLD = youden_threshold(val["label"], val["prob"])
print(f"CNN decision threshold (chosen on validation): {THRESHOLD:.3f}")

Metrics with 95% confidence intervals

In [ ]:
def metrics(y, p, threshold):
    y, p = np.asarray(y), np.asarray(p)
    yhat = (p >= threshold).astype(int)
    two_classes = len(np.unique(y)) == 2
    return {
        "AUC": roc_auc_score(y, p) if two_classes else np.nan,
        "Balanced acc": balanced_accuracy_score(y, yhat) if two_classes else np.nan,
        "Sensitivity": recall_score(y, yhat, pos_label=1, zero_division=np.nan),
        "Specificity": recall_score(y, yhat, pos_label=0, zero_division=np.nan),
        "Accuracy": accuracy_score(y, yhat),
    }

def bootstrap(y, p, threshold, n_boot=N_BOOT):
    """Resample subjects (within each class) to show how much each metric could move."""
    y, p = np.asarray(y), np.asarray(p)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    draws = []
    for _ in range(n_boot):
        idx = np.concatenate([RNG.choice(pos, len(pos)), RNG.choice(neg, len(neg))])
        draws.append(metrics(y[idx], p[idx], threshold))
    draws = pd.DataFrame(draws)
    return pd.DataFrame({
        "estimate": pd.Series(metrics(y, p, threshold)),
        "ci_low": draws.quantile(0.025),
        "ci_high": draws.quantile(0.975),
    }).round(3)

cnn_results = bootstrap(ev["label"], ev["prob"], THRESHOLD)
cnn_results